In [29]:
import duckdb
import pandas as pd

pd.set_option('display.max_rows', 120)
pd.set_option('display.max_colwidth', 80)

# Vis 1a uses the per-domain DB (smaller, faster)
transport_con = duckdb.connect('transportation.gov/cdxj.duckdb', read_only=True)

# Vis 1b — uncomment when full DB is available locally
# full_con = duckdb.connect('data/eot.duckdb', read_only=True)

TARGET_DOMAINS = [
    'usda.gov', 'commerce.gov', 'defense.gov', 'ed.gov', 'energy.gov',
    'hhs.gov', 'dhs.gov', 'hud.gov', 'doi.gov', 'justice.gov',
    'dol.gov', 'state.gov', 'transportation.gov', 'treasury.gov', 'va.gov',
]

# Base domain CASE expression (reused in Vis 1b)
DOMAIN_CASE = "\n".join(
    f"    WHEN host = '{d}' OR ends_with(host, '.{d}') THEN '{d}'"
    for d in TARGET_DOMAINS
)
BASE_DOMAIN_EXPR = f"CASE\n{DOMAIN_CASE}\n    ELSE 'other'\n  END"

print('Connected.')

Connected.


In [30]:
transport_con.sql("SELECT COUNT(*) FROM eot_captures").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       787021 │
└──────────────┘



In [ ]:
# Total captures per year
# NOTE: no 2012 data in the per-domain DB, so 2012 is missing here
transport_con.sql("""
    SELECT crawl_year, COUNT(*) AS captures
    FROM eot_captures
    GROUP BY 1 ORDER BY 1
""").show()

┌────────────┬──────────┐
│ crawl_year │ captures │
│  varchar   │  int64   │
├────────────┼──────────┤
│ 2004       │        8 │
│ 2008       │      185 │
│ 2016       │   199393 │
│ 2020       │   587435 │
└────────────┴──────────┘



In [23]:
# host, year, count
transport_con.sql("""
    SELECT host, crawl_year, COUNT(*) AS n
    FROM eot_captures
    GROUP BY 1, 2
    ORDER BY n DESC
""").show(max_rows=50)

┌────────────────────────────┬────────────┬────────┐
│            host            │ crawl_year │   n    │
│          varchar           │  varchar   │ int64  │
├────────────────────────────┼────────────┼────────┤
│ data.transportation.gov    │ 2020       │ 503223 │
│ www.transportation.gov     │ 2016       │ 179119 │
│ www.transportation.gov     │ 2020       │  77070 │
│ data.transportation.gov    │ 2016       │  20274 │
│ www7.transportation.gov    │ 2020       │   5694 │
│ datahub.transportation.gov │ 2020       │   1433 │
│ www.transportation.gov     │ 2008       │    140 │
│ transportation.gov         │ 2008       │     45 │
│ transportation.gov         │ 2020       │     15 │
│ www.transportation.gov     │ 2004       │      8 │
├────────────────────────────┴────────────┴────────┤
│ 10 rows                                3 columns │
└──────────────────────────────────────────────────┘



In [33]:
# Position 1 strings — ranked per year
seg1 = transport_con.sql("""
    SELECT
        COALESCE(NULLIF(path_seg_1, ''), '(root)') AS seg1,
        crawl_year,
        COUNT(*) AS n
    FROM eot_captures
    GROUP BY 1, 2
""").df()

seg1_pivot = seg1.pivot(index='seg1', columns='crawl_year', values='n').fillna(0).astype(int)
seg1_pivot['total'] = seg1_pivot.sum(axis=1)
seg1_pivot = seg1_pivot.sort_values('total', ascending=False)
seg1_pivot

crawl_year,2004,2008,2016,2020,total
seg1,,,,,
browse,0,0,1428,499251,500679
sites,0,0,28311,14833,43144
osdbu,0,0,36559,2569,39128
feedback,0,0,15957,12807,28764
node,0,0,17243,2871,20114
...,...,...,...,...,...
careers_entrylevel_small.jpg,0,0,1,0,1
careers_midlevel_small.jpg,0,0,1,0,1
cetera,0,0,1,0,1


In [34]:
seg1_pivot.head(50)

crawl_year,2004,2008,2016,2020,total
seg1,,,,,
browse,0,0,1428,499251,500679
sites,0,0,28311,14833,43144
osdbu,0,0,36559,2569,39128
feedback,0,0,15957,12807,28764
node,0,0,17243,2871,20114
comment,0,0,16280,0,16280
print,0,0,5542,7132,12674
OData.svc,0,0,9823,0,9823
taxonomy,0,0,2276,5514,7790


In [17]:
seg1_pivot[seg1_pivot[['2004', '2008']].sum(axis=1) > 0][['2004', '2008', 'total']].sort_values('2004', ascending=False)


crawl_year,2004,2008,total
seg1,,,
robots.txt,4,6,3455
(root),4,6,82
text,0,3,3
1.3.7,0,2,2
wtinit.js,0,2,2
wtbase.js,0,2,2
dotstyles.css,0,2,2
DCS.dcsqry,0,3,3
DCS.dcssip,0,3,3


In [18]:
seg1_pivot[seg1_pivot[['2016']].sum(axis=1) > 0][['2016', 'total']].sort_values('2016', ascending=False)


crawl_year,2016,total
seg1,,
osdbu,36559,39128
sites,28311,43144
node,17243,20114
comment,16280,16280
feedback,15957,28764
...,...,...
Jon%20Moss.jpg,1,1
Lowestfare.com,1,1
mtp,1,1


In [9]:
# Position 2 strings — ranked per year
seg2 = transport_con.sql("""
    SELECT
        COALESCE(NULLIF(path_seg_1, ''), '(root)') AS seg1,
        COALESCE(NULLIF(path_seg_2, ''), '(none)') AS seg2,
        crawl_year,
        COUNT(*) AS n
    FROM eot_captures
    GROUP BY 1, 2, 3
""").df()

seg2_pivot = seg2.pivot_table(index=['seg1', 'seg2'], columns='crawl_year', values='n',
                              aggfunc='sum').fillna(0).astype(int)
seg2_pivot['total'] = seg2_pivot.sum(axis=1)
seg2_pivot = seg2_pivot.sort_values('total', ascending=False)
seg2_pivot

crawl_year                      2004  2008   2016    2020   total
seg1      seg2                                                   
browse    (none)                   0     0   1249  499251  500500
sites     dot.gov                  0     0  19147   12705   31852
osdbu     procurement-forecast     0     0  27188    2139   29327
feedback  (none)                   0     0  15957   12807   28764
comment   reply                    0     0  16181       0   16181
...                              ...   ...    ...     ...     ...
OData.svc 9ew8-3qvq(421L)          0     0      1       0       1
          9ew8-3qvq(422L)          0     0      1       0       1
          9ew8-3qvq(423L)          0     0      1       0       1
          9ew8-3qvq(424L)          0     0      1       0       1
%22       (none)                   0     0      0       1       1

[39013 rows x 5 columns]

In [10]:
# Position 3 strings — ranked per year
seg3 = transport_con.sql("""
    SELECT
        COALESCE(NULLIF(path_seg_1, ''), '(root)') AS seg1,
        COALESCE(NULLIF(path_seg_2, ''), '(none)') AS seg2,
        COALESCE(NULLIF(path_seg_3, ''), '(none)') AS seg3,
        crawl_year,
        COUNT(*) AS n
    FROM eot_captures
    GROUP BY 1, 2, 3, 4
""").df()

seg3_pivot = seg3.pivot_table(index=['seg1', 'seg2', 'seg3'], columns='crawl_year', values='n',
                              aggfunc='sum').fillna(0).astype(int)
seg3_pivot['total'] = seg3_pivot.sum(axis=1)
seg3_pivot = seg3_pivot.sort_values('total', ascending=False)
seg3_pivot

,,crawl_year,2004,2008,2016,2020,total
seg1,seg2,seg3,,,,,
browse,(none),(none),0,0,1249,499251,500500
sites,dot.gov,files,0,0,19018,12637,31655
feedback,(none),(none),0,0,15957,12807,28764
osdbu,procurement-forecast,summary,0,0,20691,2139,22830
print,123751,(none),0,0,5393,5174,10567
...,...,...,...,...,...,...,...
node,38206,(none),0,0,1,0,1
Railroads,American-Class-1-Total-Miles-2010-present-,convert.cityValue,0,0,1,0,1
node,382,(none),0,0,1,0,1


---
## Vis 1b — All 15 Domains: URL Structural Breakdown

**Requires `full_con` — uncomment in setup cell when `data/eot.duckdb` is available locally.**

For each domain: top 50 and bottom 50 strings at each path position, plus filenames and extensions.

In [14]:
def top_bottom(df, n=50):
    """Return top N and bottom N rows from a DataFrame sorted by 'total' desc."""
    df = df.sort_values('total', ascending=False)
    top = df.head(n)
    # Bottom N excludes rows already shown in top
    bottom = df.tail(n) if len(df) > 2 * n else df.iloc[n:]
    return top, bottom

In [12]:
# Position 1 strings — top 50 / bottom 50 per domain
seg1_all = full_con.sql(f"""
    SELECT
        {BASE_DOMAIN_EXPR} AS base_domain,
        COALESCE(NULLIF(path_seg_1, ''), '(root)') AS seg1,
        COUNT(*) AS total
    FROM eot_captures
    GROUP BY 1, 2
""").df()

for domain in TARGET_DOMAINS:
    subset = seg1_all[seg1_all['base_domain'] == domain].drop(columns='base_domain')
    top, bottom = top_bottom(subset)
    print(f"\n{'='*60}")
    print(f"  {domain} — path_seg_1 ({len(subset)} unique strings)")
    print(f"{'='*60}")
    print(f"\nTop 50:")
    print(top.to_string(index=False))
    if len(bottom):
        print(f"\nBottom {len(bottom)}:")
        print(bottom.to_string(index=False))

NameError: name 'full_con' is not defined

In [ ]:
# Position 2 strings — top 50 / bottom 50 per domain
seg2_all = full_con.sql(f"""
    SELECT
        {BASE_DOMAIN_EXPR} AS base_domain,
        COALESCE(NULLIF(path_seg_2, ''), '(none)') AS seg2,
        COUNT(*) AS total
    FROM eot_captures
    GROUP BY 1, 2
""").df()

for domain in TARGET_DOMAINS:
    subset = seg2_all[seg2_all['base_domain'] == domain].drop(columns='base_domain')
    top, bottom = top_bottom(subset)
    print(f"\n{'='*60}")
    print(f"  {domain} — path_seg_2 ({len(subset)} unique strings)")
    print(f"{'='*60}")
    print(f"\nTop 50:")
    print(top.to_string(index=False))
    if len(bottom):
        print(f"\nBottom {len(bottom)}:")
        print(bottom.to_string(index=False))

In [ ]:
# Position 3 strings — top 50 / bottom 50 per domain
seg3_all = full_con.sql(f"""
    SELECT
        {BASE_DOMAIN_EXPR} AS base_domain,
        COALESCE(NULLIF(path_seg_3, ''), '(none)') AS seg3,
        COUNT(*) AS total
    FROM eot_captures
    GROUP BY 1, 2
""").df()

for domain in TARGET_DOMAINS:
    subset = seg3_all[seg3_all['base_domain'] == domain].drop(columns='base_domain')
    top, bottom = top_bottom(subset)
    print(f"\n{'='*60}")
    print(f"  {domain} — path_seg_3 ({len(subset)} unique strings)")
    print(f"{'='*60}")
    print(f"\nTop 50:")
    print(top.to_string(index=False))
    if len(bottom):
        print(f"\nBottom {len(bottom)}:")
        print(bottom.to_string(index=False))

In [ ]:
# Filenames — top 50 / bottom 50 per domain
filenames = full_con.sql(f"""
    SELECT
        {BASE_DOMAIN_EXPR} AS base_domain,
        COALESCE(NULLIF(regexp_extract(url_path, '/([^/]+)$', 1), ''), '(root)') AS filename,
        COUNT(*) AS total
    FROM eot_captures
    GROUP BY 1, 2
""").df()

for domain in TARGET_DOMAINS:
    subset = filenames[filenames['base_domain'] == domain].drop(columns='base_domain')
    top, bottom = top_bottom(subset)
    print(f"\n{'='*60}")
    print(f"  {domain} — filenames ({len(subset)} unique)")
    print(f"{'='*60}")
    print(f"\nTop 50:")
    print(top.to_string(index=False))
    if len(bottom):
        print(f"\nBottom {len(bottom)}:")
        print(bottom.to_string(index=False))

In [ ]:
# File extensions — top 50 / bottom 50 per domain
extensions = full_con.sql(f"""
    SELECT
        {BASE_DOMAIN_EXPR} AS base_domain,
        COALESCE(NULLIF(lower(regexp_extract(url_path, '\\.([a-zA-Z0-9]+)$', 1)), ''), '(none)') AS ext,
        COUNT(*) AS total
    FROM eot_captures
    GROUP BY 1, 2
""").df()

for domain in TARGET_DOMAINS:
    subset = extensions[extensions['base_domain'] == domain].drop(columns='base_domain')
    top, bottom = top_bottom(subset)
    print(f"\n{'='*60}")
    print(f"  {domain} — extensions ({len(subset)} unique)")
    print(f"{'='*60}")
    print(f"\nTop 50:")
    print(top.to_string(index=False))
    if len(bottom):
        print(f"\nBottom {len(bottom)}:")
        print(bottom.to_string(index=False))

In [ ]:
transport_con.close()
# full_con.close()